# Geometric EEG SSL — Pretrain Notebook (GPU)

**Purpose:** Pretrain all 5 variants on PhysioNet MI. Long-running GPU work.

**Prereq:** Run `colab_download.ipynb` first (or at least section 4a — PhysioNet
MI is the only dataset pretraining needs).

Each pretrain cell **auto-resumes** from the latest checkpoint on Drive if
one exists, or starts fresh otherwise. Safe to re-run after a disconnect.

**Before running:** Runtime → Change runtime type → GPU.

## 0. GPU check

In [ ]:
import subprocess, sys
result = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
                        capture_output=True, text=True)
if result.returncode == 0:
    print('GPU:', result.stdout.strip())
else:
    print('WARNING: no GPU detected — set Runtime → Change runtime type → GPU')
    sys.exit(1)

## 1. Install dependencies

In [ ]:
%%capture
!pip install mne moabb scikit-learn pyyaml scipy

## 2. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_ROOT = '/content/drive/MyDrive/geometric_eeg_ssl'
os.makedirs(DRIVE_ROOT, exist_ok=True)

# MNE data cached here — avoids re-downloading across sessions
MNE_DATA_DIR = f'{DRIVE_ROOT}/mne_data'
os.makedirs(MNE_DATA_DIR, exist_ok=True)
os.environ['MNE_DATA'] = MNE_DATA_DIR

# Checkpoints saved here
CKPT_ROOT = f'{DRIVE_ROOT}/runs'
os.makedirs(CKPT_ROOT, exist_ok=True)
print(f'Drive mounted. Checkpoints → {CKPT_ROOT}')
# Read preprocessing cache built by colab_download.ipynb.
CACHE_ROOT = f'{DRIVE_ROOT}/cache'
os.environ['EEG_CACHE_DIR'] = CACHE_ROOT
if not os.path.isdir(CACHE_ROOT):
    print(f'WARNING: no cache at {CACHE_ROOT}. '
          'Loaders will fall back to ~15 min preprocessing per run. '
          'Run colab_download.ipynb section 5 to build caches once.')
else:
    print(f'Preprocessing cache → {CACHE_ROOT}')


## 2b. Keep Colab alive while your laptop sleeps

Colab sessions are tied to the browser tab that owns them. If the laptop sleeps
or the network drops, the WebSocket dies and the runtime can be reclaimed even
though GPU compute is still busy. The training loop saves a checkpoint every 10
epochs, so a disconnect costs at most ~10 epochs — but it's better to avoid one
entirely. Three layers of protection, applied together:

**1. Stop the laptop from sleeping.**
- **macOS:** open Terminal and run `caffeinate -dis &` before closing the lid.
  Or System Settings → Battery → "Prevent automatic sleeping on power adapter
  when the display is off" + plug in.
- **Windows:** Settings → System → Power → Screen and sleep → set "When plugged
  in, put my device to sleep" to *Never*.

**2. Suppress Colab's idle prompt (cell below).**
Colab pops up a "Are you still here?" dialog after ~90 min of UI inactivity.
The JS snippet below auto-clicks the connect button every minute.

**3. Trust the resume cell.**
Each pretrain section has its own RESUME cell that finds the latest checkpoint
and restarts from there.

In [ ]:
from IPython.display import display, Javascript
display(Javascript('''
function ClickConnect() {
  const btn = document.querySelector("colab-connect-button");
  if (btn && btn.shadowRoot) {
    const inner = btn.shadowRoot.querySelector("#connect");
    if (inner) inner.click();
  }
  console.log("colab keep-alive ping " + new Date().toLocaleTimeString());
}
if (window._colabKeepAlive) clearInterval(window._colabKeepAlive);
window._colabKeepAlive = setInterval(ClickConnect, 60000);
console.log("colab keep-alive armed (60s interval)");
'''))
print('Keep-alive armed. Re-run this cell after any browser refresh.')

## 3. Clone repo

In [ ]:
import os, sys
REPO_DIR = '/content/geometric-eeg-ssl'
if not os.path.exists(REPO_DIR):
    !git clone https://github.com/tianxin-scu/geometric-eeg-ssl.git {REPO_DIR}
else:
    !git -C {REPO_DIR} pull
sys.path.insert(0, REPO_DIR)
sys.path.insert(0, f'{REPO_DIR}/src')
print('Repo ready at', REPO_DIR)

## 4. Verify PhysioNet MI is cached

In [ ]:
import os
EEGBCI_ROOT = os.path.join(MNE_DATA_DIR, 'MNE-eegbci-data', 'files', 'eegmmidb', '1.0.0')
if not os.path.isdir(EEGBCI_ROOT):
    raise RuntimeError(
        f'PhysioNet MI not found at {EEGBCI_ROOT}. '
        'Run colab_download.ipynb section 4a first.'
    )
n_subj = len([d for d in os.listdir(EEGBCI_ROOT) if d.startswith('S')])
print(f'PhysioNet MI: {n_subj} subject directories cached.')

## 5. Pretrain — G1 (geometric, score-only bias)

~7–10 hrs on T4, ~2–3 hrs on A100. Checkpoints saved every 10 epochs to Drive.

In [ ]:
import os, glob

G1_CKPT_DIR = f'{CKPT_ROOT}/g1_full'
os.makedirs(G1_CKPT_DIR, exist_ok=True)
os.environ['MNE_DATA'] = MNE_DATA_DIR

# Auto-resumes from the latest checkpoint if one exists; starts fresh otherwise.
_mode = 'resuming' if glob.glob(f'{G1_CKPT_DIR}/epoch_*.pt') else 'starting fresh'
print(f'G1 (geometric, score-bias): {_mode}')

!python -u {REPO_DIR}/scripts/pretrain.py \
    --config {REPO_DIR}/configs/pretrain/geometric_g1.yaml \
    --ckpt-dir {G1_CKPT_DIR} \
    --device cuda \
    --resume latest \
    2>&1 | tee -a {G1_CKPT_DIR}/train_log.txt

## 6. Pretrain — G2 (geometric, value-modulation)

Same duration estimate as G1. Run after G1 completes, or in a separate session.

In [ ]:
import os, glob

G2_CKPT_DIR = f'{CKPT_ROOT}/g2_full'
os.makedirs(G2_CKPT_DIR, exist_ok=True)
os.environ['MNE_DATA'] = MNE_DATA_DIR

# Auto-resumes from the latest checkpoint if one exists; starts fresh otherwise.
_mode = 'resuming' if glob.glob(f'{G2_CKPT_DIR}/epoch_*.pt') else 'starting fresh'
print(f'G2 (geometric, value-modulation): {_mode}')

!python -u {REPO_DIR}/scripts/pretrain.py \
    --config {REPO_DIR}/configs/pretrain/geometric_g2.yaml \
    --ckpt-dir {G2_CKPT_DIR} \
    --device cuda \
    --resume latest \
    2>&1 | tee -a {G2_CKPT_DIR}/train_log.txt

## 7. Pretrain — G3 (geometric, score + value)

In [ ]:
import os, glob

G3_CKPT_DIR = f'{CKPT_ROOT}/g3_full'
os.makedirs(G3_CKPT_DIR, exist_ok=True)
os.environ['MNE_DATA'] = MNE_DATA_DIR

# Auto-resumes from the latest checkpoint if one exists; starts fresh otherwise.
_mode = 'resuming' if glob.glob(f'{G3_CKPT_DIR}/epoch_*.pt') else 'starting fresh'
print(f'G3 (geometric, score + value): {_mode}')

!python -u {REPO_DIR}/scripts/pretrain.py \
    --config {REPO_DIR}/configs/pretrain/geometric_g3.yaml \
    --ckpt-dir {G3_CKPT_DIR} \
    --device cuda \
    --resume latest \
    2>&1 | tee -a {G3_CKPT_DIR}/train_log.txt

## 8. Pretrain — Transductive Codex

In [ ]:
import os, glob

CODEX_CKPT_DIR = f'{CKPT_ROOT}/codex_full'
os.makedirs(CODEX_CKPT_DIR, exist_ok=True)
os.environ['MNE_DATA'] = MNE_DATA_DIR

# Auto-resumes from the latest checkpoint if one exists; starts fresh otherwise.
_mode = 'resuming' if glob.glob(f'{CODEX_CKPT_DIR}/epoch_*.pt') else 'starting fresh'
print(f'Transductive Codex: {_mode}')

!python -u {REPO_DIR}/scripts/pretrain.py \
    --config {REPO_DIR}/configs/pretrain/transductive_codex.yaml \
    --ckpt-dir {CODEX_CKPT_DIR} \
    --device cuda \
    --resume latest \
    2>&1 | tee -a {CODEX_CKPT_DIR}/train_log.txt

## 9. Pretrain — Channel-Independent baseline

No geometric attention, no codex. Matches G1 budget. Required for E7.

In [ ]:
import os, glob

CHIND_CKPT_DIR = f'{CKPT_ROOT}/chind_full'
os.makedirs(CHIND_CKPT_DIR, exist_ok=True)
os.environ['MNE_DATA'] = MNE_DATA_DIR

# Auto-resumes from the latest checkpoint if one exists; starts fresh otherwise.
_mode = 'resuming' if glob.glob(f'{CHIND_CKPT_DIR}/epoch_*.pt') else 'starting fresh'
print(f'Channel-Independent: {_mode}')

!python -u {REPO_DIR}/scripts/pretrain.py \
    --config {REPO_DIR}/configs/pretrain/channel_independent.yaml \
    --ckpt-dir {CHIND_CKPT_DIR} \
    --device cuda \
    --resume latest \
    2>&1 | tee -a {CHIND_CKPT_DIR}/train_log.txt

## 9b. Verify checkpoint inventory

Run after all pretrains finish to confirm what's available before eval.

In [ ]:
import glob, os

VARIANTS = [
    ('G1',                 f'{CKPT_ROOT}/g1_full'),
    ('G2',                 f'{CKPT_ROOT}/g2_full'),
    ('G3',                 f'{CKPT_ROOT}/g3_full'),
    ('Transductive Codex', f'{CKPT_ROOT}/codex_full'),
    ('Channel-Indep',      f'{CKPT_ROOT}/chind_full'),
]

print(f"{'Variant':<25} {'Latest checkpoint'}")
print('-' * 60)
for label, ckpt_dir in VARIANTS:
    ckpts = sorted(glob.glob(f'{ckpt_dir}/epoch_*.pt'))
    if ckpts:
        latest = os.path.basename(ckpts[-1])
        n = len(ckpts)
        print(f'  {label:<23} {latest}  ({n} saved)')
    else:
        print(f'  {label:<23} (no checkpoints)')

## Done

All 5 variants pretrained. Switch to `colab_experiment.ipynb` to run
the E1/E2/E5/E7 eval suite (uses checkpoints from Drive; you can keep the
same GPU runtime or start a new one).